In [1]:
import lightgbm as lgb
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [2]:
RANDOM_STATE = 12345

# Load data

In [3]:
breast_cancer = load_breast_cancer(as_frame=True)
breast_cancer_df = breast_cancer.frame
feature_cols = [i.replace(' ', '_') for i in breast_cancer.feature_names]
target_cols = ['target']
breast_cancer_df.columns = feature_cols + target_cols
breast_cancer_df.head()

,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,...,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [4]:
X = breast_cancer_df[feature_cols].values
y = breast_cancer_df[target_cols].squeeze()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=RANDOM_STATE)

print('Train', X_train.shape, y_train.shape)
print('Valid', X_valid.shape, y_valid.shape)
print('Test', X_test.shape, y_test.shape)

Train (455, 30) (455,)
Valid (57, 30) (57,)
Test (57, 30) (57,)


# LGBM

In [5]:
def print_tree(tree):
    for i in tree:
        i['threshold'] = round(i['threshold'], 3)
        i['split_gain'] = round(i['split_gain'], 3)
        i['value'] = round(i['value'], 3)
        i['weight'] = round(i['weight'], 3)
        
        if not math.isnan(float(i['split_gain'])):
            message = f"[{i['split_feature']} {i['decision_type']} {i['threshold']}], gain={i['split_gain']}"
        else:
            message = f"leaf={i['value']}"

        message += f', weight={i['weight']}, count={i['count']}'

        tab = '    '*(i['node_depth']-1)
        print(f"{tab}{message}")

In [6]:
train_data = lgb.Dataset(X_train, label=y_train, feature_name=feature_cols)
valid_data = lgb.Dataset(X_valid, label=y_valid, feature_name=feature_cols)
test_data = lgb.Dataset(X_test, label=y_test, feature_name=feature_cols)

In [7]:
params = {
    'boosting_type': 'gbdt',       # Gradient Boosting Decision Tree
    'objective': 'binary',         # binary log loss classification
    'n_estimators': 1,             # Number of learners
    'num_leaves': 4,               # Max tree leaves for learners
}

classifier = lgb.train(train_set=train_data, params=params)

[LightGBM] [Info] Number of positive: 285, number of negative: 170
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000210 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4542
[LightGBM] [Info] Number of data points in the train set: 455, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.626374 -> initscore=0.516691
[LightGBM] [Info] Start training from score 0.516691


In [8]:
tree_df = classifier.trees_to_dataframe()
tree = tree_df[tree_df['tree_index'] == 0].to_dict(orient='records')
print_tree(tree)

[worst_perimeter <= 105.95], gain=330.089, weight=106.484, count=455
    [worst_concave_points <= 0.134], gain=19.21, weight=64.826, count=277
        leaf=0.673, weight=60.146, count=257
        leaf=0.463, weight=4.681, count=20
    [mean_concave_points <= 0.049], gain=31.15, weight=41.657, count=178
        leaf=0.516, weight=5.617, count=24
        leaf=0.263, weight=36.041, count=154


In [9]:
used_cols = tree_df.loc[~tree_df['split_feature'].isna(), 'split_feature'].tolist()
used_col_idxs = [feature_cols.index(i) for i in used_cols]

print(f'used_cols = {used_cols}')
print(f'used_col_idxs = {used_col_idxs}')

used_cols = ['worst_perimeter', 'worst_concave_points', 'mean_concave_points']
used_col_idxs = [22, 27, 7]


In [10]:
raw_pred = classifier.predict(X_train, raw_score=True)
prob_pred = classifier.predict(X_train)

In [11]:
print(f'idx = {0}')
print(f'raw_pred = {raw_pred[0]}')
print(f'prob_pred = {prob_pred[0]}')

idx = 0
raw_pred = 0.2629169319936248
prob_pred = 0.5653532026220233


# SHAP
## TreeExplainer
1. Extracts the tree structure from model. It parses out, for every tree in the ensemble:
   - Which feature is split on at each internal node
    - The threshold value for the split
    - The leaf values (predictions)
    - How many training samples passed through each node/leaf (this is used to estimate conditional expectations)
2. What output of the model should be explained.  **model_output** parameter: [“raw”, “probability” and “log_loss”]  
   By default model_output="raw", it explain "raw" the raw predicted value (sum of all tree outputs before any final transform).  
   You can control this with the model_output parameter (e.g. "probability" for classifiers)
3. Sets up the background dataset for expectations.  
   When a background data is provided, the `interventional` approach will be used, otherwise the `tree_path_dependent` approach will be used.
    - `interventional`: computes conditional expectations by actually replacing feature values with values from that background set and averaging. This gives a stricter Shapley-value guarantee (true marginal expectations, not path-dependent ones) but is more computationally expensive
   - `tree_path_dependent`: computed conditional expectations by following the proportion of training samples that go left/right at each split
   

### 1. TreeExplainer without background data

In [12]:
explainer = shap.TreeExplainer(classifier, model_output="raw")
shap_values = explainer(X_train)

In [13]:
print('SHAP Value of sample 0')
dict(zip(used_cols, shap_values.values[0, used_col_idxs]))

SHAP Value of sample 0


{'worst_perimeter': np.float64(-0.17062871091713883),
 'worst_concave_points': np.float64(-0.059398865631166996),
 'mean_concave_points': np.float64(-0.02374623313713272)}

#### Manual calculation

In [14]:
idx = 0
sample = X_train[idx, used_col_idxs]
sample = dict(zip(used_cols, sample))

In [15]:
# Extracts the tree structure from model 
# [f22_worst_perimeter <= 105.95]
#     Yes: [f27_worst_concave_points <= 0.134]
#         L0: Yes: leaf=0.673, weight=60.146, count=257
#         L3: No: leaf=0.463, weight=4.681, count=20
#     No: [f7_mean_concave_points <= 0.049]
#         L1: Yes: leaf=0.516, weight=5.617, count=24
#         L2 No: leaf=0.263, weight=36.041, count=154

# leaf_value, leaf_weight
L0, wL0 = 0.6730146045357606, 60.145635545253754
L3, wL3 = 0.46269177292345715, 4.680594205856322
L1, wL1 = 0.5161037963665026, 5.616713047027587
L2, wL2 = 0.2629169319936248, 36.04057538509369

w_left  = wL0 + wL3   # cover of the whole worst_perimeter(f22)<=105.95 branch
w_right = wL1 + wL2   # cover of the whole worst_perimeter(f22)>105.95 branch

In [16]:
# Compute SHAP value for each feature.

f_        = (L0*wL0 + L3*wL3 + L1*wL1 + L2*wL2) / (w_left + w_right)
f_22      = (L1*wL1 + L2*wL2) / w_right                         # 22 known -> go right, 7 unknown
f_27      = (L3*w_left + (L1*wL1+L2*wL2)) / (w_left + w_right)  # 27 known(No) -> use FULL left branch weight
f_7       = ((L0*wL0+L3*wL3) + L2*w_right) / (w_left + w_right) # 7 known(No) -> use FULL right branch weight
f_22_27   = (L1*wL1 + L2*wL2) / w_right                         # 22 known -> 27 irrelevant on this side
f_22_7    = L2                                                  # both known -> deterministic leaf
f_27_7    = (L3*w_left + L2*w_right) / (w_left + w_right)       # 22 unknown, weight by FULL branch covers
f_22_27_7 = L2

# w = |S|!(M-|S|-1)!/M!
# |S|=0, w0=2/6
# |S|=1, w1=1/6
# |S|=2, w2=2/6
w0, w1, w2 = 2/6, 1/6, 2/6

# sum[ w * [f(S U {i}) - f(S)] ]
shap_22 = w0*(f_22-f_) + w1*(f_22_27-f_27) + w1*(f_22_7-f_7) + w2*(f_22_27_7-f_27_7)
shap_27 = w0*(f_27-f_) + w1*(f_22_27-f_22) + w1*(f_27_7-f_7) + w2*(f_22_27_7-f_22_7)
shap_7  = w0*(f_7-f_)  + w1*(f_22_7-f_22)  + w1*(f_27_7-f_27) + w2*(f_22_27_7-f_22_27)

print(f'shap_f22_mean_concave_points = {shap_22}')
print(f'shap_f27_worst_concave_points = {shap_27}')
print(f'shap_f7_mean_concave_points = {shap_7}')

shap_f22_mean_concave_points = -0.17062871091713894
shap_f27_worst_concave_points = -0.05939886563116699
shap_f7_mean_concave_points = -0.023746233137132717


### 2. TreeExplainer with background data

In [17]:
# To use all samples, set max_samples=455 when initializing the masker.
masker = shap.maskers.Independent(X_train, max_samples=len(X_train))

explainer = shap.TreeExplainer(classifier, data=masker, model_output="raw")
shap_values = explainer(X_train)

In [18]:
base_values = float(shap_values.base_values[0])
print(f'base_values = {base_values}')

base_values = 0.5166907416790634


In [19]:
print('SHAP Value of sample 0')
dict(zip(used_cols, shap_values.values[0, used_col_idxs]))

SHAP Value of sample 0


{'worst_perimeter': np.float64(-0.11201967118860601),
 'worst_concave_points': np.float64(-0.05939885594032623),
 'mean_concave_points': np.float64(-0.08235529333680541)}

#### Manual calculation

In [20]:
idx = 0
sample = X_train[idx, used_col_idxs]
sample = dict(zip(used_cols, sample))
sample

{'worst_perimeter': np.float64(220.8),
 'worst_concave_points': np.float64(0.2688),
 'mean_concave_points': np.float64(0.1878)}

In [21]:
background = X_train[:, used_col_idxs] 

In [22]:
# leaf_value
L0 = 0.6730146045357606
L3 = 0.46269177292345715
L1 = 0.5161037963665026
L2 = 0.2629169319936248

def leaf_value(v22, v27, v7):
    """Deterministically walk the tree given a full (real or substituted) feature vector."""
    if v22 <= 105.95:
        return L0 if v27 <= 0.134 else L3
    else:
        return L1 if v7 <= 0.049 else L2

def f_interventional(S):
    """
    Interventional f(S): for features in S, use the sample's own value.
    For features NOT in S, substitute in each background row's value in turn
    (an actual intervention, not a cover-weighted branch average),
    then average the resulting leaf value over the whole background set.
    """
    total = 0.0
    for z in background:
        v22 = sample['worst_perimeter']      if 'worst_perimeter'      in S else z[0]
        v27 = sample['worst_concave_points'] if 'worst_concave_points' in S else z[1]
        v7  = sample['mean_concave_points']  if 'mean_concave_points'  in S else z[2]
        total += leaf_value(v22, v27, v7)
    return total / len(background)

In [23]:
# Compute f(S) for every subset S, this time by intervention/substitution
# using the background data instead of the tree's own cover statistics.

f_        = f_interventional(set())
f_22      = f_interventional({'worst_perimeter'})
f_27      = f_interventional({'worst_concave_points'})
f_7       = f_interventional({'mean_concave_points'})
f_22_27   = f_interventional({'worst_perimeter', 'worst_concave_points'})
f_22_7    = f_interventional({'worst_perimeter', 'mean_concave_points'})
f_27_7    = f_interventional({'worst_concave_points', 'mean_concave_points'})
f_22_27_7 = f_interventional({'worst_perimeter', 'worst_concave_points', 'mean_concave_points'})

# Shapley weights are unchanged - only how f(S) is computed changes between the two approaches
# w = |S|!(M-|S|-1)!/M!
w0, w1, w2 = 2/6, 1/6, 2/6

shap_22 = w0*(f_22-f_) + w1*(f_22_27-f_27) + w1*(f_22_7-f_7) + w2*(f_22_27_7-f_27_7)
shap_27 = w0*(f_27-f_) + w1*(f_22_27-f_22) + w1*(f_27_7-f_7) + w2*(f_22_27_7-f_22_7)
shap_7  = w0*(f_7-f_)  + w1*(f_22_7-f_22)  + w1*(f_27_7-f_27) + w2*(f_22_27_7-f_22_27)

print(f'base value f_    = {f_}')
print(f'shap_f22_worst_perimeter       = {shap_22}')
print(f'shap_f27_worst_concave_points  = {shap_27}')
print(f'shap_f7_mean_concave_points    = {shap_7}')

base value f_    = 0.5166907416790631
shap_f22_worst_perimeter       = -0.1122978836256994
shap_f27_worst_concave_points  = -0.05939886563116639
shap_f7_mean_concave_points    = -0.08207706042857088


### 3. TreeExplainer with background data and probability output

In [24]:
# To use all samples, set max_samples=455 when initializing the masker.
masker = shap.maskers.Independent(X_train, max_samples=len(X_train))

explainer = shap.TreeExplainer(classifier, data=masker, model_output="probability")
shap_values = explainer(X_train)

In [25]:
base_values = float(shap_values.base_values[0])
print(f'base_values = {base_values}')

base_values = 0.6253776302889621


In [26]:
print('SHAP Value of sample 0')
dict(zip(used_cols, shap_values.values[0, used_col_idxs]))

SHAP Value of sample 0


{'worst_perimeter': np.float64(-0.02649146521922749),
 'worst_concave_points': np.float64(-0.014024154979368918),
 'mean_concave_points': np.float64(-0.0195088100226235)}

#### Manual calculation

In [27]:
idx = 0
sample = X_train[idx, used_col_idxs]
sample = dict(zip(used_cols, sample))
sample

{'worst_perimeter': np.float64(220.8),
 'worst_concave_points': np.float64(0.2688),
 'mean_concave_points': np.float64(0.1878)}

In [28]:
background = X_train[:, used_col_idxs]

In [29]:
# LightGBM's leaf values are raw margins (logits); 
# the binary objective turns them into a probability with a sigmoid.
# So for model_output="probability" the value function becomes:
#
#     f(S) = E_z[ sigmoid( raw(x_S ; z_~S) ) ]
#
# i.e. apply the sigmoid to EACH background substitution's leaf value first,
# then average. The transform is non-linear, so it must live INSIDE the
# background average, not be applied to the averaged raw score afterwards.

# leaf_value (raw margins, same tree as before)
L0 = 0.6730146045357606
L3 = 0.46269177292345715
L1 = 0.5161037963665026
L2 = 0.2629169319936248

def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))

def leaf_value(v22, v27, v7):
    """Deterministically walk the tree and return the raw (logit) leaf value."""
    if v22 <= 105.95000:
        return L0 if v27 <= 0.13370	else L3
    else:
        return L1 if v7 <= 0.04923 else L2

def f_interventional_prob(S):
    """
    Interventional f(S) for the PROBABILITY output: for features in S use the
    sample's own value; for features not in S substitute each background row's
    value in turn, push the resulting raw leaf value through the sigmoid, and
    average those probabilities over the whole background set.
    """
    total = 0.0
    for z in background:
        v22 = sample['worst_perimeter']      if 'worst_perimeter'      in S else z[0]
        v27 = sample['worst_concave_points'] if 'worst_concave_points' in S else z[1]
        v7  = sample['mean_concave_points']  if 'mean_concave_points'  in S else z[2]
        total += sigmoid(leaf_value(v22, v27, v7))
    return total / len(background)

# Compute f(S) for every subset S on the probability scale.
f_        = f_interventional_prob(set())
f_22      = f_interventional_prob({'worst_perimeter'})
f_27      = f_interventional_prob({'worst_concave_points'})
f_7       = f_interventional_prob({'mean_concave_points'})
f_22_27   = f_interventional_prob({'worst_perimeter', 'worst_concave_points'})
f_22_7    = f_interventional_prob({'worst_perimeter', 'mean_concave_points'})
f_27_7    = f_interventional_prob({'worst_concave_points', 'mean_concave_points'})
f_22_27_7 = f_interventional_prob({'worst_perimeter', 'worst_concave_points', 'mean_concave_points'})

# Shapley weights are unchanged - only how f(S) is computed changes.
# w = |S|!(M-|S|-1)!/M!
w0, w1, w2 = 2/6, 1/6, 2/6

shap_22 = w0*(f_22-f_) + w1*(f_22_27-f_27) + w1*(f_22_7-f_7) + w2*(f_22_27_7-f_27_7)
shap_27 = w0*(f_27-f_) + w1*(f_22_27-f_22) + w1*(f_27_7-f_7) + w2*(f_22_27_7-f_22_7)
shap_7  = w0*(f_7-f_)  + w1*(f_22_7-f_22)  + w1*(f_27_7-f_27) + w2*(f_22_27_7-f_22_27)

print(f'base value f_    = {f_}')                      # matches base_values (mean predicted prob)
print(f'shap_f22_worst_perimeter       = {shap_22}')
print(f'shap_f27_worst_concave_points  = {shap_27}')
print(f'shap_f7_mean_concave_points    = {shap_7}')

# Efficiency check: base value + all SHAP values == the model's predicted probability.
print(f'\nf_ + sum(shap) = {f_ + shap_22 + shap_27 + shap_7}')
print(f'prob_pred[idx] = {prob_pred[idx]}')

base value f_    = 0.6253776302889591
shap_f22_worst_perimeter       = -0.02651631060377598
shap_f27_worst_concave_points  = -0.013704399516276628
shap_f7_mean_concave_points    = -0.01980371754688544

f_ + sum(shap) = 0.5653532026220209
prob_pred[idx] = 0.5653532026220233


#### Reproducing SHAP's numbers exactly

The cell above computes the *exact* Shapley values of the value function $f(S) = \mathbb{E}_z\big[\sigma(\text{raw}(x_S; z_{\sim S}))\big]$. That is the textbook answer, and it gets close, but it is **not** bit-for-bit what `shap` prints for `model_output="probability"`.

The reason is how `shap`'s interventional algorithm handles the non-linear link. For **each** background reference $z$ it:

1. computes the exact **raw** (logit) Shapley contributions $\phi^{\text{raw}}_i$, which by efficiency sum to $\text{raw}(x) - \text{raw}(z) = \text{leaf}_x - \text{leaf}_z$; then
2. rescales every contribution by the **secant slope** of the link between the two endpoints,
$$s = \frac{\sigma(\text{leaf}_x) - \sigma(\text{leaf}_z)}{\text{leaf}_x - \text{leaf}_z},$$

and finally **averages** $\phi^{\text{raw}}_i \cdot s$ over all references. So the transformed per-reference change $\sigma(\text{leaf}_x) - \sigma(\text{leaf}_z)$ is split among features *in proportion to their raw contributions*, rather than by running a fresh Shapley game directly on the probabilities. This reproduces `shap` to ~1e-9.

In [30]:
import itertools

# Reuses leaf_value, sigmoid, L0..L3 and the exact thresholds from the cell above.

def raw_shap_for_reference(z):
    """Exact RAW (logit) Shapley contributions for the (sample, reference z) pair."""
    def raw(S):
        return leaf_value(
            sample['worst_perimeter']      if 'worst_perimeter'      in S else z[0],
            sample['worst_concave_points'] if 'worst_concave_points' in S else z[1],
            sample['mean_concave_points']  if 'mean_concave_points'  in S else z[2])
    phi = {c: 0.0 for c in used_cols}
    for c in used_cols:
        rest = [x for x in used_cols if x != c]
        for r in range(len(rest) + 1):
            for combo in itertools.combinations(rest, r):
                S = set(combo)
                w = math.factorial(len(S)) * math.factorial(2 - len(S)) / 6   # |S|!(M-|S|-1)!/M!, M=3
                phi[c] += w * (raw(S | {c}) - raw(S))
    return phi

leaf_x = leaf_value(sample['worst_perimeter'], sample['worst_concave_points'], sample['mean_concave_points'])

# For each reference: raw Shapley contributions, rescaled by the secant slope of
# the sigmoid link, then averaged over the whole background set.
shap_prob = {c: 0.0 for c in used_cols}
for z in background:
    leaf_z = leaf_value(z[0], z[1], z[2])
    if leaf_x == leaf_z:
        s = sigmoid(leaf_x) * (1 - sigmoid(leaf_x))                    # link derivative when endpoints coincide
    else:
        s = (sigmoid(leaf_x) - sigmoid(leaf_z)) / (leaf_x - leaf_z)    # secant slope of the link
    phi = raw_shap_for_reference(z)
    for c in used_cols:
        shap_prob[c] += phi[c] * s / len(background)

print('Manual (secant-rescaled), matches shap TreeExplainer(model_output="probability"):')
for c in used_cols:
    print(f'shap_{c:22s} = {shap_prob[c]}')

# Compare directly against shap's own values from the cell higher up.
print('\nshap library values:')
for c, v in dict(zip(used_cols, shap_values.values[0, used_col_idxs])).items():
    print(f'shap_{c:22s} = {float(v)}')

Manual (secant-rescaled), matches shap TreeExplainer(model_output="probability"):
shap_worst_perimeter        = -0.026491461699873174
shap_worst_concave_points   = -0.01402415726739033
shap_mean_concave_points    = -0.01950880869967514

shap library values:
shap_worst_perimeter        = -0.02649146521922749
shap_worst_concave_points   = -0.014024154979368918
shap_mean_concave_points    = -0.0195088100226235
